# Приложение А. Сбор и кодирование постов ВКонтакте

**Скрипт выполняет три задачи:**  
1. Сбор постов из ВКонтакте-сообществ рекреационных пространств Мурманска через VK API  
2. Автоматическое кодирование текстов постов по шести аналитическим блокам  
3. Проверку значимости различий между периодами (χ²-тест) и сохранение в `posts_coded.xlsx`

## 1. Установка зависимостей и импорт библиотек

In [ ]:
!pip install vk_api openpyxl --quiet

import re
import time
import pandas as pd
from datetime import datetime, timezone
from scipy.stats import chi2_contingency


## 2. Параметры запуска

Укажите ваш VK Access Token и пути к файлам перед запуском.

In [ ]:
VK_ACCESS_TOKEN = "vk1.a.MvbW-v7YTeI7948nBwW22zvuo5wxIEL_W_oFZBt4HShHZUR5K-jcm11s5qSRFSW4QKifMHiXJIMmR9w1svFJRNa2MX8FgB_32LlWNRW4y63nNTIysmI8JUILs_Ye-n1yjQA9AThRNXYP8RDVHXaHHFPUAWVNU0DJ1oYzSnqY7xL4u_sa3oAZWt1I7jxvbevLN6lwVQdiTCEnMon5DkhrJw""
PLACES_FILE     = "Murmansk_ca_base.xlsx"   # база мест с колонкой vk_url
POSTS_CSV_OUT   = "posts_raw.csv"
CODED_XLSX_OUT  = "posts_coded.xlsx"

# Экспериментальный период — полярная ночь (ноябрь–январь)
EXP_MONTHS  = {11, 12, 1}
# Контрольный период — осень вне полярной ночи (сентябрь–октябрь + февраль)
CTRL_MONTHS = {9, 10, 2}
TARGET_MONTHS = EXP_MONTHS | CTRL_MONTHS

# Собираем посты не старше 3 лет
CUTOFF_TS = int(datetime(2023, 1, 1, tzinfo=timezone.utc).timestamp())


## 3. Загрузка базы мест

Читает `Murmansk_ca_base.xlsx`, извлекает `screen_name` из столбца `vk_url`.

In [ ]:
def load_places(path):
    df = pd.read_excel(path, header=0)
    df.columns = ["num", "name", "mentions", "category",
                  "lat", "lon", "address", "description", "vk_url"]
    df = df[df["vk_url"].notna()].copy()
    df["screen_name"] = df["vk_url"].str.extract(r"vk\.com/(.+)$")
    df = df[df["screen_name"].notna()].copy()
    df = df[~df["screen_name"].str.contains(r"\?", na=False)].copy()
    return df

places = load_places(PLACES_FILE)
print(f"Мест с валидными ВКонтакте-пабликами: {len(places)}")
print(places[["name", "category", "screen_name"]].to_string(index=False))


## 4. Сбор постов через VK API

Для каждого сообщества запрашивает посты постранично (по 100 за запрос) и останавливается, как только дата поста уходит за `CUTOFF_TS`. Сохраняются только посты из `TARGET_MONTHS`. Лимит запросов: 3 в секунду (задержка 0.34 с).

In [ ]:
def fetch_posts_vk(screen_names, token):
    import vk_api
    vk_session = vk_api.VkApi(token=token)
    vk = vk_session.get_api()
    records = []

    for sn in screen_names:
        print(f" → {sn}", end=" ")
        offset, stop, sn_count = 0, False, 0

        while not stop:
            try:
                resp  = vk.wall.get(domain=sn, count=100,
                                    offset=offset, filter="owner")
                items = resp.get("items", [])
                if not items:
                    break

                for post in items:
                    post_ts = post["date"]
                    if post_ts < CUTOFF_TS:
                        stop = True
                        break
                    dt    = datetime.fromtimestamp(post_ts)
                    month = dt.month
                    if month not in TARGET_MONTHS:
                        continue
                    records.append({
                        "screen_name": sn,
                        "post_id":     post["id"],
                        "datetime":    dt,
                        "hour":        dt.hour,
                        "weekday":     dt.weekday(),
                        "month":       month,
                        "year":        dt.year,
                        "likes":       post.get("likes",    {}).get("count", 0),
                        "reposts":     post.get("reposts",  {}).get("count", 0),
                        "comments":    post.get("comments", {}).get("count", 0),
                        "views":       post.get("views",    {}).get("count", 0),
                        "text":        post.get("text", ""),
                        "is_repost":   "copy_history" in post,
                    })
                    sn_count += 1

                offset += 100
                time.sleep(0.34)

            except Exception as e:
                print(f"\n  Ошибка {sn}: {e}")
                break

        print(f"→ {sn_count} постов")

    return pd.DataFrame(records)


screen_names = places["screen_name"].dropna().tolist()
df_posts = fetch_posts_vk(screen_names, VK_ACCESS_TOKEN)
df_posts.to_csv(POSTS_CSV_OUT, index=False, encoding="utf-8-sig")

exp_n  = df_posts["month"].isin(EXP_MONTHS).sum()
ctrl_n = df_posts["month"].isin(CTRL_MONTHS).sum()
print(f"\nСобрано постов всего : {len(df_posts)}")
print(f"  Экспериментальный (ноябрь–январь): {exp_n}")
print(f"  Контрольный (сент–окт + февраль):  {ctrl_n}")


## 5. Фильтрация технических постов

Удаляются: пустые посты, посты из одной URL, репосты без текста, посты без кириллицы, а также системные уведомления ВКонтакте (смена фото, обложки, названия и т. п.).

In [ ]:
_TECH_PATTERNS = [
    r"обновил.{0,10}фотографи",
    r"обновил.{0,10}обложк",
    r"сменил.{0,10}аватар",
    r"добавил.{0,10}фотографи",
    r"изменил.{0,10}название",
    r"создал.{0,10}событи",
]

def is_technical(row):
    text = str(row.get("text", "")).strip()
    if len(text) == 0: return True
    if re.match(r"^https?://\S+$", text): return True
    if row.get("is_repost", False) and len(text) < 10: return True
    if not re.findall(r"[а-яёa-z]{3,}", text.lower()): return True
    for p in _TECH_PATTERNS:
        if re.search(p, text.lower()): return True
    return False


## 6. Нормализация текста

Текст приводится к нижнему регистру; удаляются URL, упоминания (@user), хэштеги и спецсимволы.

In [ ]:
def clean_text(text):
    if pd.isna(text): return ""
    t = str(text).lower()
    t = re.sub(r"http\S+", " ", t)
    t = re.sub(r"@\w+",    " ", t)
    t = re.sub(r"#\w+",    " ", t)
    t = re.sub(r"[^а-яёa-z\s\-]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


## 7. Кодовые книги

Шесть аналитических блоков: ритмика/сенсорика, функция поста, фокус, нарратив пространства, механика воздействия, тональность.  
Для блоков «Функция» и «Тон» применяется система приоритетов: при срабатывании нескольких паттернов фиксируется код с наибольшим приоритетом.

In [ ]:
# ── Блок 1: Ритмика и сенсорика (дихотомические переменные) ──────────────
CODEBOOK_RHYTHMIC = {
    "SYNC": [
        r"утр(о|ом|енний)", r"с утра",
        r"днём", r"дневной", r"обед",
        r"вечер(ом|ний|а)", r"ночь[ю]?", r"ночной",
        r"понедельник", r"вторник", r"среда", r"четверг",
        r"в пятниц", r"в субботу", r"в воскресенье",
        r"по выходным", r"в выходные", r"на выходн",
        r"после пар", r"между парами", r"после занятий", r"после лекци",
        r"после работы", r"с работы", r"рабочий день",
        r"после тяжёл", r"после тяжел",
    ],
    "CONTEXT_LIGHT": [
        r"полярная ночь", r"полярн(ой|ую) ноч", r"заполярь",
        r"темнот[ауе]", r"тьм[ае]", r"темно за окном",
        r"без солнц", r"солнце не встаёт", r"солнца нет",
        r"короткий день", r"световой день", r"мало света",
        r"рассвет", r"закат", r"сумерки", r"сумрак", r"серость",
    ],
    "CONTEXT_WEATHER": [
        r"мороз", r"морозн", r"лютый мороз",
        r"холодн", r"холод(?!ильник)",
        r"метел[ья]", r"вьюга", r"буран", r"пурга",
        r"снег(?!овик)", r"снежн", r"сугробы?", r"занесло",
        r"ветер", r"ветрен", r"порывист",
        r"голол[её]д", r"скольз", r"наледь",
        r"плохая погода", r"непогод",
    ],
    "CONTEXT_SEASON": [
        r"зима", r"зимни(й|е|м|ми|х)", r"зимой", r"зимнее время",
        r"зимний сезон", r"в это время года", r"холодное время",
        r"северная зима",
    ],
    "SENSORY_LIGHT": [
        r"светло", r"светл[ое]",
        r"яркий свет", r"много света",
        r"т[её]плый свет", r"мягкий свет", r"приглушённый свет",
        r"огни", r"огоньк[и]", r"фонар",
        r"гирлянды?", r"свеч[аи]", r"подсветк",
        r"камин", r"живой огонь",
    ],
    "SENSORY_WARM": [
        r"тепло(?!ход)", r"согреться", r"согревайтесь", r"согревающ",
        r"прогреться", r"отогреться",
        r"горячий чай", r"горячий кофе", r"горячий шоколад",
        r"горяч(ее|ие)", r"тёплый напиток",
        r"укрыться от холода", r"спрятаться от холода",
        r"не мёрзнуть", r"не замёрзнуть",
    ],
    "SENSORY_BODY": [
        r"устал[иа]", r"усталость", r"переутомлени", r"измучен",
        r"сонливость", r"сонн", r"хочется спать", r"клонит ко сну",
        r"бодрость", r"бодр", r"взбодрит", r"зарядиться",
        r"энерги[яию]", r"заряд",
        r"расслабит", r"расслабл", r"релакс",
        r"отдохнут", r"отдыха", r"восстанов", r"перезарядит",
    ],
}

# ── Блок 2: Функция поста (доминирующая) ──────────────────────────────────
CODEBOOK_FUNCTION = {
    "PRM": [r"скидк", r"акци[яию]", r"спецпредложени", r"промокод",
            r"закажи", r"брониру", r"успей"],
    "ENG": [r"розыгрыш", r"конкурс", r"напиши.{0,10}коммент",
            r"делитес", r"ответьте", r"угадайте", r"участвуй", r"голосу[ий]"],
    "IMG": [r"атмосфер", r"вдохновени", r"стиль жизн",
            r"эстетик", r"место силы", r"особое место"],
    "COM": [r"друзья", r"вместе", r"наши гости",
            r"ждём вас", r"ждем вас", r"комьюнити", r"сообществ"],
    "INF": [r"режим работы", r"открыт с", r"закрыт", r"работаем",
            r"анонс", r"состоится", r"расписани", r"новое меню"],
}
FUNCTION_PRIORITY = ["PRM", "ENG", "IMG", "COM", "INF"]

# ── Блок 3: Фокус поста (мультикод, не более 3) ───────────────────────────
CODEBOOK_FOCUS = {
    "PROD":    [r"кофе", r"напит", r"меню", r"десерт", r"блюд[оа]",
               r"еда", r"коктейл", r"чай", r"выпеч", r"бургер"],
    "EVENT":   [r"концерт", r"вечерин[кеу]", r"лекци", r"квиз",
               r"мероприяти", r"выступлени", r"показ", r"встреч[ае]"],
    "SPACE":   [r"интерьер", r"пространств", r"зал", r"террас",
               r"дизайн", r"атмосфер"],
    "PEOPLE":  [r"бариста", r"команда", r"гости", r"музыкант",
               r"ведущ", r"артист"],
    "OFFER":   [r"скидк", r"бонус", r"акци", r"подарок",
               r"спецпредложени", r"бесплатн"],
    "ROUTINE": [r"провести вечер", r"заглян", r"посидеть",
               r"выходные", r"отдохнуть", r"после работы", r"после пар"],
}

# ── Блок 4: Нарратив пространства (мультикод, не более 2) ─────────────────
CODEBOOK_NARRATIVE = {
    "PLACE_THIRD":     [r"встрет", r"провести вечер", r"посидеть",
                        r"отдохнуть", r"заглянуть", r"зайти", r"побыть"],
    "PLACE_EXPERIENCE":[r"атмосфер", r"эмоци", r"впечатлени",
                        r"настроени", r"ощущени", r"почувству"],
    "PLACE_UTILITY":   [r"быстро", r"удобн", r"рядом", r"близко",
                        r"с собой", r"доставк", r"завтрак", r"обед"],
    "PLACE_EVENTFUL":  [r"событи", r"афиш", r"программ",
                        r"концерт", r"вечер", r"мероприяти"],
    "PLACE_COMMUNITY": [r"свои", r"вместе", r"сообществ",
                        r"наши гости", r"комьюнити", r"семья"],
    "PLACE_LOCAL":     [r"мурманск", r"север", r"северный",
                        r"локальн", r"арктик", r"полярн", r"заполярь"],
    "PLACE_ESCAPE":    [r"убежищ", r"укрыться", r"спрятаться",
                        r"забыть про", r"отвлечься", r"перезагруз",
                        r"оазис", r"островок", r"другой мир"],
    "PLACE_CONTRAST":  [r"снаружи.{0,20}внутри", r"за окном.{0,20}у нас",
                        r"на улице.{0,20}здесь", r"мороз.{0,20}тепло",
                        r"темно.{0,20}светло", r"холодно.{0,20}уютно"],
    "PLACE_EXOTIC":    [r"уникальн", r"только здесь", r"нигде больше",
                        r"настоящий север", r"аутентичн"],
    "PLACE_HOME":      [r"как дома", r"домашн", r"по-домашнему",
                        r"второй дом", r"своё место"],
}

# ── Блок 5: Механика воздействия (мультикод, не более 2) ─────────────────
CODEBOOK_MECHANIC = {
    "CALL_DIRECT":  [r"приходите", r"ждём вас", r"ждем вас",
                     r"бронируйте", r"заказывайте", r"успейте",
                     r"не пропустите", r"записывайтесь"],
    "CALL_SOFT":    [r"попробуйте", r"загляните", r"приходи",
                     r"приглашаем", r"будем рады"],
    "BENEFIT":      [r"выгодн", r"удобн", r"скидк", r"бесплатн",
                     r"спеццен", r"дёшев", r"дешев"],
    "EMOTION":      [r"приятн", r"любим", r"радост", r"счасть",
                     r"вдохновени", r"тепл", r"уют"],
    "SCARCITY":     [r"только сегодня", r"последн", r"ограничен",
                     r"мало мест", r"успей", r"до конца"],
    "NOVELTY":      [r"новинк", r"новое меню", r"впервые",
                     r"обновили", r"скоро открытие"],
    "RITUAL":       [r"утро с", r"вечер с", r"каждую", r"каждый",
                     r"традиц", r"всегда", r"как обычно"],
    "DIALOGUE":     [r"расскажите", r"как вы", r"а вы", r"напишите нам",
                     r"что думаете", r"ваше мнение", r"делитесь"],
    "SOCIAL_PROOF": [r"любим(ое|ый|ая) место", r"лучш", r"культов",
                     r"популярн", r"гости в восторге", r"рекомендуют"],
    "TRADITION":    [r"с \d{4} года", r"уже \d+ лет",
                     r"классика города", r"легенда", r"проверено временем"],
    "SENSORY_HOOK": [r"ароматный", r"свежий", r"горячий", r"тёплый",
                     r"мягкий", r"уютный свет", r"живой огонь", r"вкуснейш"],
}

# ── Блок 6: Тональность (доминирующий тон) ────────────────────────────────
CODEBOOK_TONE = {
    "PUSH_HARD":  [r"скидк", r"акци", r"успей", r"только сегодня",
                   r"бронируйте", r"заказывайте", r"не пропустите"],
    "PUSH_SOFT":  [r"попробуйте", r"загляните", r"ждём вас",
                   r"ждем вас", r"приглашаем", r"приходите"],
    "PULL_AESTH": [r"атмосфер", r"эстетик", r"вдохнов", r"настроени"],
    "PULL_BELONG":[r"вместе", r"свои", r"наши гости", r"друзья",
                   r"сообществ", r"комьюнити"],
    "PULL_VALUE": [r"мурманск", r"локальн", r"арктик", r"северный",
                   r"место силы", r"наш город"],
}
TONE_PRIORITY = ["PUSH_HARD", "PUSH_SOFT", "PULL_AESTH", "PULL_BELONG", "PULL_VALUE"]

# Переменные блока 1, по которым проводится χ²-тест
RHYTHMIC_VARS = [
    "SYNC", "CONTEXT_LIGHT", "CONTEXT_WEATHER", "CONTEXT_SEASON",
    "SENSORY_LIGHT", "SENSORY_WARM", "SENSORY_BODY",
]


## 8. Вспомогательные функции кодирования

In [ ]:
def match_any(text, patterns):
    for p in patterns:
        if re.search(p, text):
            return 1
    return 0

def detect_multi(text, codebook, max_codes=None):
    found = [code for code, pats in codebook.items()
             if match_any(text, pats)]
    if max_codes:
        found = found[:max_codes]
    return ", ".join(found) if found else ""

def detect_dominant(text, codebook, priority):
    for code in priority:
        if code in codebook and match_any(text, codebook[code]):
            return code
    return ""

def mark_period(month):
    if pd.isna(month): return ""
    m = int(month)
    if m in EXP_MONTHS:  return "experimental"
    if m in CTRL_MONTHS: return "control"
    return "other"


## 9. Кодирование постов

Последовательно применяет все шесть кодовых книг к очищенному тексту. Технические посты удаляются на этом же этапе.

In [ ]:
def code_posts(df):
    df = df.copy()
    df["text_clean"] = df["text"].apply(clean_text)

    # Фильтрация технических постов
    df["IS_TECHNICAL"] = df.apply(is_technical, axis=1)
    n_before = len(df)
    df = df[~df["IS_TECHNICAL"]].drop(columns=["IS_TECHNICAL"]).copy()
    print(f"Технических постов: {n_before - len(df)} из {n_before} ({len(df)} осталось)")

    t = df["text_clean"]

    # Временны́е метки
    dt = pd.to_datetime(df["datetime"])
    df["hour"]    = dt.dt.hour
    df["weekday"] = dt.dt.weekday
    df["month"]   = dt.dt.month
    df["TIME_EVENING"] = (df["hour"]    >= 17).astype(int)
    df["TIME_WEEKEND"] = (df["weekday"] >= 5).astype(int)
    df["period"]       = df["month"].apply(mark_period)

    # Блок 1: ритмика и сенсорика
    for code, pats in CODEBOOK_RHYTHMIC.items():
        df[code] = t.apply(lambda x: match_any(x, pats))

    # Блок 2–6
    df["main_function"]  = t.apply(
        lambda x: detect_dominant(x, CODEBOOK_FUNCTION, FUNCTION_PRIORITY))
    df["focus_codes"]    = t.apply(
        lambda x: detect_multi(x, CODEBOOK_FOCUS, max_codes=3))
    df["narrative_codes"]= t.apply(
        lambda x: detect_multi(x, CODEBOOK_NARRATIVE, max_codes=2))
    df["mechanic_codes"] = t.apply(
        lambda x: detect_multi(x, CODEBOOK_MECHANIC, max_codes=2))
    df["tone"]           = t.apply(
        lambda x: detect_dominant(x, CODEBOOK_TONE, TONE_PRIORITY))

    return df


df_coded = code_posts(df_posts)


## 10. χ²-тест по ритмическим и сенсорным кодам

Для каждой дихотомической переменной блока 1 строится таблица сопряжённости 2×2 (период × наличие кода) и вычисляется χ² Пирсона.

In [ ]:
def chi2_periods(df):
    subset  = df[df["period"].isin(["experimental", "control"])].copy()
    exp_tot = (subset["period"] == "experimental").sum()
    ctrl_tot= (subset["period"] == "control").sum()
    results = []
    for var in RHYTHMIC_VARS:
        if var not in subset.columns:
            continue
        ct = pd.crosstab(subset["period"], subset[var])
        if ct.shape != (2, 2):
            continue
        chi2, p, _, _ = chi2_contingency(ct)
        exp_n  = subset.loc[subset["period"] == "experimental", var].sum()
        ctrl_n = subset.loc[subset["period"] == "control",       var].sum()
        results.append({
            "variable":    var,
            "exp_n":       int(exp_n),
            "exp_%":       round(exp_n  / exp_tot  * 100, 1) if exp_tot  else 0,
            "ctrl_n":      int(ctrl_n),
            "ctrl_%":      round(ctrl_n / ctrl_tot * 100, 1) if ctrl_tot else 0,
            "chi2":        round(chi2, 3),
            "p_value":     round(p, 4),
            "sig":         ("***" if p < 0.001 else
                            "**"  if p < 0.01  else
                            "*"   if p < 0.05  else
                            "~"   if p < 0.1   else "n.s."),
        })
    return pd.DataFrame(results)


df_chi2 = chi2_periods(df_coded)
print("\nχ²-тест (экспериментальный период vs контрольный):\n")
print(df_chi2.to_string(index=False))


## 11. Сохранение результатов

Файл `posts_coded.xlsx` содержит четыре листа:  
- `Coded_Posts` — основной датасет (9 230 постов × 32 переменных)  
- `Chi2_Results` — таблица χ²-теста  
- `Freq_Function` — частоты доминирующей функции поста  
- `By_Year_Period` — разбивка по годам и периодам

In [ ]:
OUTPUT_COLS = [
    "screen_name", "post_id", "datetime", "year",
    "hour", "weekday", "month", "period",
    "TIME_EVENING", "TIME_WEEKEND", "SYNC",
    "CONTEXT_LIGHT", "CONTEXT_WEATHER", "CONTEXT_SEASON",
    "SENSORY_LIGHT", "SENSORY_WARM", "SENSORY_BODY",
    "main_function", "focus_codes", "narrative_codes",
    "mechanic_codes", "tone",
    "likes", "reposts", "comments", "views", "text",
]
cols = [c for c in OUTPUT_COLS if c in df_coded.columns]

with pd.ExcelWriter(CODED_XLSX_OUT, engine="openpyxl") as writer:
    df_coded[cols].to_excel(writer, sheet_name="Coded_Posts", index=False)
    if len(df_chi2):
        df_chi2.to_excel(writer, sheet_name="Chi2_Results", index=False)
    df_coded["main_function"].value_counts().reset_index().rename(
        columns={"index": "main_function", 0: "count"}
    ).to_excel(writer, sheet_name="Freq_Function", index=False)
    if "year" in df_coded.columns:
        (df_coded.groupby(["year", "period"])
                 .size().reset_index(name="n")
                 .to_excel(writer, sheet_name="By_Year_Period", index=False))

print(f"Сохранено: {CODED_XLSX_OUT}")
print(f"Обработано постов: {len(df_coded)}")
